# ADV Clear — the full pipeline

**What this notebook does:** takes a PDF of scanned receipts and produces the JSON that the
webapp needs, using two AI models that both run on your own machine.

Two models, because each is good at a different job:

| | Model | Its job |
|---|---|---|
| **Stage 1** | typhoon-ocr1.5-3b | Look at the picture and type out the Thai text it sees |
| **Stage 2** | qwen3:4b | Read that text and pick out the date, seller, total, tax, etc. |

```
receipt image  ──►  Stage 1  ──►  Thai text  ──►  Stage 2  ──►  JSON for the webapp
```

Typhoon is excellent at reading Thai handwriting but can only produce plain text — it cannot
produce JSON. So a second model turns its text into the fields the form needs.

**To use this notebook:** run the cells from top to bottom. Change `PDF` in the setup cell to
whichever file you want to test.

---
## 1. Setup

Loads our two helper files and sets the options. Nothing is read or sent anywhere yet — this cell
just prepares things.

**About `ROOT`:** the code lives in `src/` and the receipts live in `data/samples/`, so this cell
first works out where the project folder is, then builds every other path from it. That way the
notebook runs the same whether the kernel started in `FA_OCR/` or somewhere else — which is the
bug that used to make this cell fail with *No module named 'ocr_pipeline'*.

**The settings you might want to change:**

- `PDF` — which file to read. Works with `.pdf`, `.png`, `.jpg`. Put new files in `data/samples/`.
- `TARGET_DIM` — how big to make the page image, in pixels. **Leave at 1500.** At exactly 1800 the
  model breaks on some pages and returns `@@@@@@`. Yes, 1800 is what Typhoon's own website
  recommends. It still breaks.
- `REPEAT_PENALTY` — stops the model repeating itself forever. **1.25, not 1.1** — below ~1.2 the
  model falls into filling a form's empty ruled boxes with `<tr><td>-</td>` rows and gets truncated
  before it reaches the totals at the bottom of the page.

`importlib.reload` is there so that if you edit `ocr_pipeline.py` or `stage2_extract.py`, your
changes apply without restarting the kernel.

In [ ]:
import importlib
import json
import sys
import time
from pathlib import Path

from IPython.display import HTML, Markdown, display

# Find the project folder by walking up from wherever the kernel started, looking for src/.
# Everything below is built from ROOT, so no path depends on the current working directory.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src" / "ocr_pipeline.py").exists()), None)
if ROOT is None:
    raise RuntimeError(f"Could not find the FA_OCR folder starting from {Path.cwd()}")
sys.path[:0] = [str(ROOT / "src"), str(ROOT / "eval")]

import ocr_pipeline
import stage2_extract
importlib.reload(ocr_pipeline)      # so edits to the .py files apply without a kernel restart
importlib.reload(stage2_extract)

from ocr_pipeline import build_prompt, collapse_empty_rows, load_pages, ocr_page, unload
from stage2_extract import assemble, extract, validate

PDF            = ROOT / "data/samples/test2.png"
STAGE1_MODEL   = "scb10x/typhoon-ocr1.5-3b"
STAGE2_MODEL   = "qwen3:4b"
STAGE2_OPTIONS = {"think": False}      # 6x faster, and the amounts stay just as accurate
TARGET_DIM     = 1500                  # do not use 1800
REPEAT_PENALTY = 1.25                  # not 1.1, see the note above
CATEGORY       = None                  # FA's account code, e.g. "5223100". None = shared prompt.

OUT = ROOT / "eval/transcripts"
OUT.mkdir(parents=True, exist_ok=True)

def looks_failed(text):
    """Spot the broken output: a short burst of one repeated character."""
    return text is None or len(text) < 80 or text.count("@") > 10

print(f"project folder : {ROOT}")
print(f"reading        : {PDF.name}   ({'found' if PDF.exists() else 'MISSING'})")

---
## 2. Stage 1 — read the pages

Goes through every page of the file, turns each one into a picture, and asks Typhoon to type out
what it sees. Saves each page's text into `eval/transcripts/` so you don't have to re-run this
slow step later.

**What you'll see:** one line per page with the time taken and how many characters came back.
Anything marked `FAILED` means the model returned garbage for that page — see section 7.

**Expect 8–90 seconds per page.** Pages with big empty tables are the slow ones.

**About the prompt:** `build_prompt()` returns Typhoon's own official instruction text. You cannot
replace it — this model was trained so tightly on that exact wording that any substitute makes it
repeat the instruction back at you instead of reading the picture. You can only add to it, and even
then it mostly ignores what you add. Stage 2's prompt, in section 4, is the editable one.

In [ ]:
prompt1 = build_prompt(figure_language="English")
transcripts = {}

for page_num, img in load_pages(PDF, TARGET_DIM):
    started = time.perf_counter()
    text = ocr_page(img, prompt1, model=STAGE1_MODEL, repeat_penalty=REPEAT_PENALTY)
    transcripts[page_num] = text
    (OUT / f"p{page_num}.txt").write_text(text, encoding="utf-8")
    flag = "   <-- FAILED, see section 7" if looks_failed(text) else ""
    print(f"page {page_num}  {img.size}  {time.perf_counter() - started:5.1f}s  "
          f"{len(text):>6} chars{flag}")

---
## 3. Check what stage 1 actually read

Shows one page's text, formatted. **Do read this** — if stage 2 gets a field wrong, the first thing
to check is whether the text was even there for it to find.

Change `PAGE` to look at a different page.

You'll notice tables come out as HTML. That is on purpose: a receipt's totals box is a table, and
keeping the table structure is how stage 2 tells "amount before VAT" apart from "VAT" apart from
"total". Flattening it to plain text would lose that.

In [ ]:
PAGE = 1
display(Markdown(transcripts[PAGE]))

---
## 4. Stage 2 — turn the text into form fields

For each page, four things happen:

0. **`collapse_empty_rows`** — throws away the blank grid. These receipt forms are pre-printed with
   ruled lines, and stage 1 transcribes every blank line as a table row: page 3 came back as 281
   rows of which 1117 cells were empty, 96% of the file. Deleting rows where *every* cell is blank
   loses nothing — measured on all ten pages, the visible text is byte-identical — and it saves
   stage 2 from reading 13,000 characters of markup to find three numbers.
1. **`extract`** — asks qwen3 to pull out the fields. It runs in "grammar mode", which means the
   model is physically prevented from producing anything other than the exact shape we asked for.
   No missing brackets, no extra fields, no explanation text.
2. **`assemble`** — builds the response the webapp expects, adds `chunkPageIndex`, and applies the
   two rules that a prompt alone could not enforce: a tax ID that isn't 13 digits (or is TEAM's
   own) becomes null, and a document number with no digit in it becomes null, because that means
   the model read the pre-printed `เลขที่ / BOOK NO.` label off an empty box.
3. **`validate`** — checks the result against your colleague's real schema file. That file is the
   only thing that counts. If this says `valid`, their code will accept it.

**Expect 10–15 seconds per page.**

**About the shape:** the contract was 22 fields per bill. Five were dropped by agreement on
2026-08-11 — `regions`, `evidence` and the three category fields — along with `sourceRegions` on
every field, because Typhoon gives no coordinates and FA picks the category before uploading.
`chunkPageIndex` was then added back on its own: it used to live inside `regions`, and without it
a four-page chunk returns four bills with nothing saying which scan each came from.

---

### Per-category rules

Set `CATEGORY` in the setup cell to one of FA's account codes and stage 2 appends that category's
extra rules to its prompt. Set it to `None` and every bill gets the shared prompt.

**To write a rule, edit [`data/category_rules.json`](data/category_rules.json).** No Python. All 82
of FA's account codes are already in there with an empty rule, which means "behave normally" — so
you only fill in the ones that actually need a twist. The file is re-read on every request, so an
edit takes effect immediately.

Stage 1 never sees the category. Typhoon's only job is to type out what's on the paper, and that
doesn't change between travel and subcontract.

Two limits worth knowing:

- **A rule cannot add a field.** The schema is closed. There is nowhere to put a "destination" or a
  "project code" — the only free-text field is `lineItems[].description`.
- **A rule is a suggestion, not a guarantee.** The model may ignore it. Run the same page with
  `CATEGORY = None` and with the code set, and compare, rather than assuming it worked.

In [ ]:
import stage2_extract as s2

rules = s2.load_category_rules()
print(f"{len(s2.load_categories())} categories known, {len(rules)} with their own rules"
      + (f": {', '.join(rules)}" if rules else " -- all using the shared prompt"))
if CATEGORY:
    print(f"this run: {CATEGORY} {s2.load_categories().get(CATEGORY, {}).get('name', '(unknown code)')}")
print()

responses = {}

for page_num, text in transcripts.items():
    if looks_failed(text):
        print(f"page {page_num}  skipped -- stage 1 gave us nothing to work with")
        continue
    clean = collapse_empty_rows(text)      # drop the blank ruled grid; keeps every filled row

    started = time.perf_counter()
    reduced = extract(clean, STAGE2_MODEL, STAGE2_OPTIONS, category=CATEGORY)
    response = assemble(reduced, page_index=page_num - 1,   # the webapp counts pages from 0
                        transcript=clean)                  # lets assemble check labels near a number
    ok, err = validate(response)
    responses[page_num] = response
    print(f"page {page_num}  {time.perf_counter() - started:5.1f}s  "
          f"{len(text):>6} -> {len(clean):<5} chars  "
          f"{'valid' if ok else 'INVALID: ' + str(err)}  "
          f"{len(response['billCandidates'])} bill(s) found")

---
## 5. The results in one table

Every page's extracted fields side by side, so you can check them against the receipts by eye.

A dash means the model found nothing and returned empty. **That is the correct behaviour** when a
value genuinely isn't on the receipt — an empty box costs FA a few seconds of typing, whereas a
confident wrong answer might get accepted without anyone noticing.

The last column is every line item's description, one per line. It's the quickest way to see
whether the model read the *contents* of the bill or only its totals box — and if a category rule
in section 7 asks for something to be kept in the description, this is where you check that it was.

In [ ]:
import html as _html

KEYS = ["documentDate", "sellerName", "sellerTaxId", "originalDocumentNumber",
        "currency", "amountBeforeVat", "vat", "originalTotal", "clearingAmount"]

def line_items(candidate):
    """Every line item's description, one per line. None if the model found no items at all."""
    found = [(item["description"]["value"] or "").strip() for item in candidate["lineItems"]]
    return "\n".join(d for d in found if d) or None

def cell(value):
    # Escaped, because these strings came out of a model reading a receipt -- a stray "<" in a
    # description would otherwise eat the rest of the table.
    if value is None:
        return "—"
    return _html.escape(str(value)).replace("\n", "<br>")

rows = [[f"p{n}"] + [c[k]["value"] for k in KEYS] + [line_items(c)]
        for n, resp in sorted(responses.items()) for c in resp["billCandidates"]]

th = "".join(f"<th style='text-align:left;padding:4px 10px'>{h}</th>"
             for h in ["page"] + KEYS + ["lineItems"])
tr = "".join("<tr style='border-top:1px solid #8884'>" + "".join(
        f"<td style='padding:4px 10px;vertical-align:top'>{cell(v)}</td>" for v in r) + "</tr>"
     for r in rows)
display(HTML(f"<table style='font:13px system-ui;border-collapse:collapse'>"
             f"<tr>{th}</tr>{tr}</table>"))

---
## 6. One complete response

This is exactly what your colleague's webapp receives for one page. Worth looking at once so you
know the shape you're both agreeing on — every field is wrapped as `{value, confidence}`, where
`confidence` is how sure the model claims to be (0 to 1).

A `null` means *the document does not say*. That is a real answer, not a failure — `vat: null` on a
market cash bill is correct, and it tells FA to go look at the paper. `vat: 0` would claim the paper
says zero, which is a different and usually wrong statement.

Watch the confidence numbers here. Within one page they tend to come back nearly identical, even
on fields that are wrong — which is the thing section 10 measures properly.

In [ ]:
if responses:
    first = sorted(responses)[0]
    print(json.dumps(responses[first], ensure_ascii=False, indent=2))

---
## 7. If pages come back as `@@@@@@`

Two things cause this, and this cell tells you which.

Ollama keeps a memory of the previous request. In a long run, each page's result can be affected by
the page before it — which is exactly what made our early measurements contradict each other.
`unload()` clears the model out between pages so each one is measured on its own.

**Slow, and only for troubleshooting.** Never do this in the real system.

**Reading the result:**

- **Same pages fail here as in section 2** → that page genuinely breaks the model. Try a different
  `TARGET_DIM`.
- **Pages that failed in section 2 now work** → it was leftover state, not the page.

In [ ]:
print("page   result on a clean model")
for page_num, img in load_pages(PDF, TARGET_DIM):
    unload(STAGE1_MODEL)
    text = ocr_page(img, prompt1, model=STAGE1_MODEL, repeat_penalty=REPEAT_PENALTY)
    print(f"  {page_num}    {'FAILED' if looks_failed(text) else f'{len(text):>6} chars'}")

---
## 8. Does it give the same answer twice?

Your colleague's contract says a retried page must produce identical output, because their queue
retries jobs. This reads the same page three times and prints a short fingerprint of each result.

**All three fingerprints the same** = good. **Different** = we still have work to do here.

Right now this does not fully pass, which is a known open item.

In [ ]:
_, img = next(load_pages(PDF, TARGET_DIM))
for i in range(3):
    text = ocr_page(img, prompt1, model=STAGE1_MODEL, repeat_penalty=REPEAT_PENALTY)
    print(f"run {i + 1}   {len(text):>6} chars   fingerprint {hash(text) & 0xffffff:06x}")

---
## 9. Score it against the answer key

Everything above tells you the pipeline *ran*. This cell is the only thing that tells you whether
it was **right**.

It needs `data/golden/golden.json` filled in — that is the list of correct answers, typed by a
human off the receipts. Instructions are in [`data/golden/cases.md`](data/golden/cases.md).
You don't have to finish all nine receipts; fill in two and you already get a real number.

What it prints, in order:

1. **`todo_report`** — how much of the answer key is still blank.
2. **`check_response`** — the things a field-by-field score cannot see: did stage 1 fail, is the
   JSON schema-valid, did it split one receipt into two, and **is confidence flat?** Flat
   confidence means the highlighting in your colleague's UI would point FA at the wrong fields.
3. **`confidence_report`** — accuracy grouped by how confident the model claimed to be. This
   number should climb down the rows. If it doesn't, confidence is decoration.
4. **`build_inputs`** — writes the two files your colleague's TypeScript scorer reads, and prints
   the command to run it. **That scorer, not anything here, is the official number.**

Stage 1 output is cached per receipt, so re-running this after editing the stage-2 prompt is fast.

In [ ]:
import harness
importlib.reload(harness)

golden = harness.load_golden()
harness.todo_report(golden)
filled = harness.scorable(golden)          # only the receipts with something typed in

if not filled["cases"]:
    print("\nThe answer key is still empty -- see data/golden/cases.md. Nothing to score yet.")
else:
    golden_responses = harness.run_cases(filled)
    print()
    harness.check_response(golden_responses)
    print()
    harness.confidence_report(golden_responses, golden)
    print()
    harness.build_inputs(golden_responses, golden)

---
## Speed benchmark: this machine against the others

Times stage 1 and stage 2 on the **same pages** on every machine, so a rented GPU can be compared
with the laptop like for like. **Run section 1 (setup) first**; no other section is needed.

- **Warms both models on one page first**, and never counts it. A cold model is slower, and its
  first answers differ.
- **Saves only timings**, no document text, to `eval/speed/<gpu>_<host>.json`. Commit that file:
  every other machine's comparison cell picks it up after `git pull`.
- **Times the notebook's own path**: stage 1, then stage 2, one page at a time. The guards the
  server adds on top cost ~0.1% (plan/07 section 1b), so they are left out. The loop re-read is not
  in this path either; a page that loops is marked, not re-read.

About 15 minutes on the laptop (15 pages).

In [ ]:
import platform, re, socket, subprocess, threading
from datetime import datetime

BENCH_FILES = ["data/samples/บิลเงินสด_ร้านค้า_5ใบ.pdf",
               "data/samples/บิลเงินสด_ร้านค้า_10ใบ.pdf"]
SPEED_DIR = ROOT / "eval/speed"
SPEED_DIR.mkdir(parents=True, exist_ok=True)

def smi(field):
    """One nvidia-smi value per GPU as floats or strings; [] when there is no NVIDIA GPU."""
    try:
        out = subprocess.run(["nvidia-smi", f"--query-gpu={field}", "--format=csv,noheader,nounits"],
                             capture_output=True, text=True, timeout=10).stdout
    except (OSError, subprocess.SubprocessError):
        return []
    vals = [v.strip() for v in out.splitlines() if v.strip()]
    try:
        return [float(v) for v in vals]
    except ValueError:
        return vals

class PeakVram:
    """Samples GPU memory in use every half second while the block runs."""
    def __enter__(self):
        self.peak_mib, self._stop = 0.0, threading.Event()
        def poll():
            while not self._stop.wait(0.5):
                used = smi("memory.used")
                if used and isinstance(used[0], float):
                    self.peak_mib = max(self.peak_mib, sum(used))
        self._thread = threading.Thread(target=poll, daemon=True)
        self._thread.start()
        return self
    def __exit__(self, *exc):
        self._stop.set()
        self._thread.join()

def cpu_name():
    try:
        return next(l.split(":", 1)[1].strip() for l in open("/proc/cpuinfo") if l.startswith("model name"))
    except (OSError, StopIteration):
        return platform.processor() or platform.machine()

def git_commit():
    try:
        return subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=ROOT,
                              capture_output=True, text=True, timeout=10).stdout.strip() or None
    except (OSError, subprocess.SubprocessError):
        return None

machine = {"gpu": " + ".join(smi("name")) or "no NVIDIA GPU",
           "vram_total_mib": sum(v for v in smi("memory.total") if isinstance(v, float)),
           "cpu": cpu_name(), "host": socket.gethostname(), "os": platform.platform()}
print(f"machine  {machine['gpu']}  ·  {machine['cpu']}  ·  {machine['host']}")

bench_prompt = build_prompt(figure_language="English")     # the same prompt section 2 uses

# Warm-up: load both models and run them once. Not counted.
started = time.perf_counter()
_, warm_img = next(load_pages(ROOT / BENCH_FILES[0], TARGET_DIM))
warm_text = ocr_page(warm_img, bench_prompt, model=STAGE1_MODEL, repeat_penalty=REPEAT_PENALTY)
extract(collapse_empty_rows(warm_text), STAGE2_MODEL, STAGE2_OPTIONS)
print(f"warm-up  {time.perf_counter() - started:.1f}s  (not counted)\n")

rows = []
with PeakVram() as vram:
    for rel in BENCH_FILES:
        pages = load_pages(ROOT / rel, TARGET_DIM)
        while True:
            t0 = time.perf_counter()
            try:
                page_num, img = next(pages)                     # rasterising happens here
            except StopIteration:
                break
            t1 = time.perf_counter()
            text = ocr_page(img, bench_prompt, model=STAGE1_MODEL, repeat_penalty=REPEAT_PENALTY)
            t2 = time.perf_counter()
            row = dict(file=Path(rel).name, page=page_num, rasterise_s=t1 - t0, stage1_s=t2 - t1,
                       stage2_s=None, stage1_chars=len(text or ""), stage1_failed=looks_failed(text),
                       stage2_error=None)
            if not row["stage1_failed"]:
                try:
                    extract(collapse_empty_rows(text), STAGE2_MODEL, STAGE2_OPTIONS, category=None)
                except Exception as e:
                    row["stage2_error"] = f"{type(e).__name__}: {e}"[:200]
                row["stage2_s"] = time.perf_counter() - t2
            rows.append(row)
            s2 = "  stage 1 FAILED, stage 2 skipped" if row["stage1_failed"] else \
                 f"  stage 2 {row['stage2_s']:5.1f}s" + ("  ERROR" if row["stage2_error"] else "")
            print(f"{row['file'][:28]:<28} p{page_num:<3} stage 1 {row['stage1_s']:5.1f}s{s2}")

result = {"machine": machine, "when": datetime.now().isoformat(timespec="seconds"),
          "commit": git_commit(), "peak_vram_mib": vram.peak_mib,
          "settings": {"stage1_model": STAGE1_MODEL, "stage2_model": STAGE2_MODEL,
                       "stage2_options": STAGE2_OPTIONS, "target_dim": TARGET_DIM,
                       "repeat_penalty": REPEAT_PENALTY, "files": BENCH_FILES},
          "pages": rows}
label = re.sub(r"[^A-Za-z0-9]+", "-", f"{machine['gpu']}_{machine['host']}").strip("-")[:80]
out_file = SPEED_DIR / f"{label}.json"
out_file.write_text(json.dumps(result, ensure_ascii=False, indent=1), encoding="utf-8")

done = [r for r in rows if r["stage2_s"] is not None]
per_page = sum(r["rasterise_s"] + r["stage1_s"] + r["stage2_s"] for r in done) / max(len(done), 1)
print(f"\n{len(rows)} pages, {len(rows) - len(done)} failed in stage 1  ·  "
      f"{per_page:.1f} s/page  ·  peak VRAM {vram.peak_mib / 1024:.1f} GB")
print(f"saved {out_file}\n  <- commit this file so other machines can compare against it")

In [ ]:
# Compares every run saved in eval/speed/. Needs only section 1 to have run.
try:
    import pandas as pd
    import matplotlib.pyplot as plt
except ImportError:
    get_ipython().run_line_magic("pip", "install -q pandas matplotlib")
    import pandas as pd
    import matplotlib.pyplot as plt

runs = [json.loads(p.read_text(encoding="utf-8")) for p in sorted((ROOT / "eval/speed").glob("*.json"))]
if not runs:
    raise SystemExit("No runs in eval/speed/ yet -- run the benchmark cell above first.")

settings = {json.dumps(r["settings"], sort_keys=True) for r in runs}
if len(settings) > 1:
    print("WARNING: these runs used different settings (models, image size or files), so they are "
          "not like for like. Compare with care.\n")

rows = []
for r in runs:
    pages = pd.DataFrame(r["pages"])
    done = pages[pages.stage2_s.notna()]
    s1, s2 = done.stage1_s.mean(), done.stage2_s.mean()
    total = (done.rasterise_s + done.stage1_s + done.stage2_s).mean()
    rows.append({"Machine": f"{r['machine']['gpu']} ({r['machine']['host']})",
                 "Measured": r["when"][:16].replace("T", " "), "Commit": r["commit"],
                 "Pages": len(pages), "Stage 1 failed": int(pages.stage1_failed.sum()),
                 "Stage 1 (s/page)": s1, "Stage 2 (s/page)": s2, "Total (s/page)": total,
                 "10-page chunk (s)": 10 * total,
                 "Peak VRAM (GB)": r["peak_vram_mib"] / 1024,
                 "GPU memory (GB)": r["machine"]["vram_total_mib"] / 1024})
table = pd.DataFrame(rows).sort_values("Total (s/page)", ascending=False, ignore_index=True)
slowest = table["Total (s/page)"].iloc[0]
table["× faster than slowest"] = slowest / table["Total (s/page)"]
display(table.style.format(precision=1, na_rep="—").hide(axis="index"))
print("The server's deadline is 600 s per chunk; '10-page chunk' shows how close each machine comes.")

# Stage 1 and stage 2 per page, stacked: the two parts of one page's time.
fig, ax = plt.subplots(figsize=(9, 0.8 * len(table) + 1.6))
fig.patch.set_facecolor("#fcfcfb"); ax.set_facecolor("#fcfcfb")
names = [m.replace(" (", "\n(") for m in table.Machine]
ax.barh(names, table["Stage 1 (s/page)"], color="#2a78d6", height=0.55, label="Stage 1 (reading)")
ax.barh(names, table["Stage 2 (s/page)"], left=table["Stage 1 (s/page)"], color="#eb6834",
        height=0.55, label="Stage 2 (fields)", edgecolor="#fcfcfb", linewidth=2)
for y, (tot, fast) in enumerate(zip(table["Total (s/page)"], table["× faster than slowest"])):
    ax.text(tot, y, f"  {tot:.1f} s" + (f"  ({fast:.1f}× faster)" if fast > 1.005 else ""),
            va="center", fontsize=9, color="#0b0b0b")
ax.set_xlabel("seconds per page", color="#52514e")
ax.set_title("Time per page, by stage", loc="left", fontweight="bold")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="x", color="#e6e5e0"); ax.set_axisbelow(True)
ax.set_xlim(0, table["Total (s/page)"].max() * 1.35)
ax.legend(frameon=False, loc="lower right", bbox_to_anchor=(1, 1), ncol=2)   # above, clear of the bars
fig.tight_layout()
plt.show()

---
## Concurrency test: several pages at once

How much more the GPU gets through when several pages are in flight together, as when several
documents arrive at once. **Run section 1 (setup) first**; nothing else is needed.

- N workers each take the next page and run stage 1 then stage 2 on it, for N = 1 to 8. The 15
  benchmark pages are run twice per level.
- Prints `OLLAMA_NUM_PARALLEL`, which caps how many pages Ollama really works on at once. Above
  it, extra pages only wait in Ollama's queue.
- Saves timings only to `eval/speed/concurrency/<gpu>_<host>.json` (a separate folder, so the
  speed compare cell above ignores it).
- This measures the **GPU**, not the server: `serving/` still takes one request at a time
  (`MAX_CONCURRENT=1`).

About 10 minutes on an RTX 5090.

In [ ]:
# ── Concurrency test: several pages in flight at once ─────────────────────────────────────────
# Needs only section 1. Simulates N documents being processed at the same time: N workers each
# take the next page and run stage 1 then stage 2 on it, exactly as the speed benchmark does.
import json, re, socket, subprocess, threading, time
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime
import pandas as pd
import matplotlib.pyplot as plt

LEVELS      = [1, 2, 3, 4, 5, 6, 8]    # pages in flight at once
BENCH_FILES = ["data/samples/บิลเงินสด_ร้านค้า_5ใบ.pdf",
               "data/samples/บิลเงินสด_ร้านค้า_10ใบ.pdf"]
REPEAT      = 2                        # run the 15 pages this many times per level
OUT_DIR     = ROOT / "eval/speed/concurrency"   # its own folder, so the speed compare cell ignores it
OUT_DIR.mkdir(parents=True, exist_ok=True)

def smi(field):
    """One nvidia-smi value per GPU, as strings; [] when there is no NVIDIA GPU."""
    try:
        out = subprocess.run(["nvidia-smi", f"--query-gpu={field}", "--format=csv,noheader,nounits"],
                             capture_output=True, text=True, timeout=10).stdout
    except (OSError, subprocess.SubprocessError):
        return []
    return [v.strip() for v in out.splitlines() if v.strip()]

class PeakVram:
    """Samples GPU memory in use every half second while the block runs."""
    def __enter__(self):
        self.peak_mib, self._stop = 0.0, threading.Event()
        def poll():
            while not self._stop.wait(0.5):
                try:
                    self.peak_mib = max(self.peak_mib, sum(float(v) for v in smi("memory.used")))
                except ValueError:
                    pass
        self._thread = threading.Thread(target=poll, daemon=True)
        self._thread.start()
        return self
    def __exit__(self, *exc):
        self._stop.set()
        self._thread.join()

def ollama_num_parallel():
    """OLLAMA_NUM_PARALLEL of the running `ollama serve`, read from /proc. None where unreadable."""
    for p in Path("/proc").glob("[0-9]*"):
        try:
            cmd = (p / "cmdline").read_bytes().split(b"\0")
            if any(c.endswith(b"ollama") for c in cmd) and b"serve" in cmd:
                env = dict(kv.split("=", 1) for kv in
                           (p / "environ").read_bytes().decode(errors="replace").split("\0") if "=" in kv)
                return env.get("OLLAMA_NUM_PARALLEL", "not set (Ollama chooses)")
        except OSError:
            continue
    return None

conc_prompt = build_prompt(figure_language="English")          # the same prompt section 2 uses
conc_pages = [(Path(f).name, n, img) for f in BENCH_FILES for n, img in load_pages(ROOT / f, TARGET_DIM)]
work = conc_pages * REPEAT

def one_page(item):
    name, num, img = item
    row = dict(file=name, page=num, stage1_s=None, stage2_s=None, total_s=None, failed=False, error=None)
    t0 = time.perf_counter()
    try:
        text = ocr_page(img, conc_prompt, model=STAGE1_MODEL, repeat_penalty=REPEAT_PENALTY)
        t1 = time.perf_counter()
        row["stage1_s"], row["failed"] = t1 - t0, looks_failed(text)
        if not row["failed"]:
            extract(collapse_empty_rows(text), STAGE2_MODEL, STAGE2_OPTIONS, category=None)
            row["stage2_s"] = time.perf_counter() - t1
    except Exception as e:
        row["error"] = f"{type(e).__name__}: {e}"[:200]
    row["total_s"] = time.perf_counter() - t0
    return row

gpu = " + ".join(smi("name")) or "no NVIDIA GPU"
num_parallel = ollama_num_parallel()
print(f"GPU {gpu}  ·  OLLAMA_NUM_PARALLEL = {num_parallel or 'unknown (could not read the server)'}")
try:
    if max(LEVELS) > int(num_parallel):
        print(f"  note: above {num_parallel} in flight, extra pages wait in Ollama's queue instead of running")
except (TypeError, ValueError):
    pass
print(f"{len(work)} pages per level, levels {LEVELS}\n")
one_page(conc_pages[0])                                          # warm-up, not counted

summary, all_rows = [], []
for n in LEVELS:
    with PeakVram() as vram:
        t0 = time.perf_counter()
        with ThreadPoolExecutor(max_workers=n) as pool:
            rows = list(pool.map(one_page, work))
        wall = time.perf_counter() - t0
    ok = pd.DataFrame([r for r in rows if r["error"] is None and not r["failed"]])
    s = {"In flight": n, "Pages": len(rows),
         "Errors": sum(r["error"] is not None for r in rows),
         "Stage 1 failed": sum(r["failed"] for r in rows),
         "Wall (s)": wall,
         "Pages / min": 60 * len(rows) / wall,
         "Each page, avg (s)": ok.total_s.mean() if len(ok) else float("nan"),
         "Each page, p95 (s)": ok.total_s.quantile(0.95) if len(ok) else float("nan"),
         "Stage 1 avg (s)": ok.stage1_s.mean() if len(ok) else float("nan"),
         "Stage 2 avg (s)": ok.stage2_s.mean() if len(ok) else float("nan"),
         "Peak VRAM (GB)": vram.peak_mib / 1024}
    s["10-page chunk (s)"] = 10 * s["Each page, avg (s)"]
    s["40-page chunk (s)"] = 40 * s["Each page, avg (s)"]
    summary.append(s)
    all_rows += [dict(r, in_flight=n) for r in rows]
    print(f"{n:>2} in flight   {s['Pages / min']:6.1f} pages/min   each page {s['Each page, avg (s)']:5.1f}s avg "
          f"/ {s['Each page, p95 (s)']:5.1f}s p95   VRAM {s['Peak VRAM (GB)']:4.1f} GB   errors {s['Errors']}")

table = pd.DataFrame(summary)
table.insert(6, "× throughput vs 1", table["Pages / min"] / table["Pages / min"].iloc[0])
display(table.round(1))                     # plain DataFrame: no jinja2 needed

best = table.loc[table["Pages / min"].idxmax()]
fits40 = table[table["40-page chunk (s)"] <= 600]
print(f"\nMost throughput: {best['Pages / min']:.0f} pages/min at {int(best['In flight'])} in flight "
      f"({best['× throughput vs 1']:.1f}× one at a time).")
print("40-page chunks inside the 600 s deadline: "
      + (f"up to {int(fits40['In flight'].max())} at once." if len(fits40) else "not at any level tested."))

label = re.sub(r"[^A-Za-z0-9]+", "-", f"{gpu}_{socket.gethostname()}").strip("-")[:80]
out_file = OUT_DIR / f"{label}.json"
out_file.write_text(json.dumps({"gpu": gpu, "host": socket.gethostname(),
                                "when": datetime.now().isoformat(timespec="seconds"),
                                "ollama_num_parallel": num_parallel, "repeat": REPEAT,
                                "files": BENCH_FILES, "levels": summary, "pages": all_rows},
                               ensure_ascii=False, indent=1), encoding="utf-8")
print(f"saved {out_file}")

# Two charts, one scale each: how much gets done, and how long each page waits.
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.2))
for ax in (a1, a2):
    ax.set_facecolor("#fcfcfb"); ax.grid(color="#e6e5e0"); ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_xticks(LEVELS); ax.set_xlabel("pages in flight at once")
    try:
        ax.axvline(int(num_parallel), ls=":", color="#52514e", lw=1)
        ax.annotate(f"OLLAMA_NUM_PARALLEL = {num_parallel}", (int(num_parallel), 1), xycoords=("data", "axes fraction"),
                    xytext=(4, -12), textcoords="offset points", fontsize=8, color="#52514e")
    except (TypeError, ValueError):
        pass
fig.patch.set_facecolor("#fcfcfb")
a1.plot(table["In flight"], table["Pages / min"], "o-", color="#2a78d6", lw=2)
a1.set_ylim(bottom=0); a1.set_title("Throughput (pages per minute)", loc="left", fontweight="bold")
a2.plot(table["In flight"], table["Each page, avg (s)"], "o-", color="#2a78d6", lw=2, label="average")
a2.plot(table["In flight"], table["Each page, p95 (s)"], "o--", color="#eb6834", lw=2, label="slowest 5%")
a2.axhline(15, ls="--", lw=1, color="#52514e")
a2.annotate("40-page chunk hits 600 s", (1, 15), xycoords=("axes fraction", "data"), xytext=(-4, 4),
            textcoords="offset points", ha="right", fontsize=8, color="#52514e")
a2.set_ylim(bottom=0); a2.set_title("Time for each page (s)", loc="left", fontweight="bold")
a2.legend(frameon=False, loc="upper left")
fig.tight_layout()
plt.show()